# ESIS-QUANT — Layer 4: Arabic Sentiment Transformer (Kaggle GPU training)

This notebook is **self-contained**: it embeds the exact tokenizer, model, and
training code from the `esis_quant` repo, so it runs on Kaggle with no repo
upload — only the training CSV.

## How to run
1. **Settings → Accelerator → GPU T4 x2** (or P100). CPU also works but is slow.
2. **Add data**: upload `sentiment_training_data.csv` (exported by
   `models/sentiment/dataset.export_training_csv`) as a Kaggle Dataset, then
   *Add Input*. Set `CSV_PATH` in the config cell to its path
   (e.g. `/kaggle/input/esis-sentiment/sentiment_training_data.csv`).
3. **Run All**.
4. Download `best_sentiment_model.pt` from the notebook **Output** and place it
   at `models/sentiment/checkpoints/best_sentiment_model.pt` in the repo. The
   local `SentimentPipeline` loads it automatically.

The checkpoint is saved with exactly the keys the repo's
`SentimentPipeline._try_load_checkpoint` expects:
`vocab_size`, `tokenizer_vocab`, `tokenizer_merges`, `model_state_dict`.

> **Data-volume note:** with only the 50 synthetic bootstrap rows, this trains a
> *correct pipeline*, not a model at target F1 (0.65–0.75). Re-export the CSV as
> real scraped news accumulates (Layer 1), re-upload, and re-run — same notebook.


In [ ]:
# ============================================================
# CONFIG  (mirrors esis_quant/config.py SENTIMENT_* — keep in sync)
# ============================================================
import os

# >>> EDIT THIS to point at your uploaded CSV <<<
CSV_PATH = "/kaggle/input/esis-sentiment/sentiment_training_data.csv"
if not os.path.exists(CSV_PATH):
    # fallbacks: a file placed in the working dir, or Kaggle's single-input guess
    for c in ["sentiment_training_data.csv", "/kaggle/working/sentiment_training_data.csv"]:
        if os.path.exists(c):
            CSV_PATH = c; break
print("Using CSV_PATH =", CSV_PATH)

SENTIMENT_VOCAB_SIZE   = 8000
SENTIMENT_SPECIAL_TOKENS = {"[PAD]": 0, "[UNK]": 1, "[CLS]": 2, "[SEP]": 3, "[MASK]": 4}
SENTIMENT_MAX_SEQ_LEN  = 128

SENTIMENT_D_MODEL      = 256
SENTIMENT_N_HEADS      = 8
SENTIMENT_N_LAYERS     = 6
SENTIMENT_D_FF         = 1024
SENTIMENT_DROPOUT      = 0.1
SENTIMENT_N_CLASSES    = 3   # Negative=0, Neutral=1, Positive=2

SENTIMENT_BATCH_SIZE   = 32
SENTIMENT_EPOCHS       = 30
SENTIMENT_LR           = 3e-4
SENTIMENT_WEIGHT_DECAY = 0.01
SENTIMENT_WARMUP_RATIO = 0.10
SENTIMENT_GRAD_CLIP_NORM = 1.0
SENTIMENT_VAL_FRACTION = 0.15

OUTPUT_CHECKPOINT = "/kaggle/working/best_sentiment_model.pt"
OUTPUT_TOKENIZER  = "/kaggle/working/sentiment_tokenizer.json"

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| torch", torch.__version__)

In [ ]:
# ============================================================
# BPE TOKENIZER  (verbatim logic from models/sentiment/tokenizer.py)
# ============================================================
import json, re, logging
from collections import Counter
logger = logging.getLogger("esis_quant.sentiment.tokenizer")

_DIACRITICS_PATTERN   = re.compile(r"[ً-ٟ]")
_ALEF_VARIANTS_PATTERN= re.compile(r"[آأإ]")   # آ أ إ -> ا
_TEH_MARBUTA_PATTERN  = re.compile(r"ة")                  # ة -> ه
_WAW_VARIANT_PATTERN  = re.compile(r"ؤ")                  # ؤ -> و
_TATWEEL_PATTERN      = re.compile(r"ـ")                  # ـ removed
_YA_VARIANTS_PATTERN  = re.compile(r"[ئى]")          # ئ ى -> ي
_KEEP_PATTERN         = re.compile(r"[^ء-ي٠-٩٪٫%.0-9\s]")
_MULTISPACE_PATTERN   = re.compile(r"\s+")

def preprocess(text: str) -> str:
    text = _DIACRITICS_PATTERN.sub("", text)
    text = _ALEF_VARIANTS_PATTERN.sub("ا", text)
    text = _TEH_MARBUTA_PATTERN.sub("ه", text)
    text = _WAW_VARIANT_PATTERN.sub("و", text)
    text = _TATWEEL_PATTERN.sub("", text)
    text = _YA_VARIANTS_PATTERN.sub("ي", text)
    text = _KEEP_PATTERN.sub("", text)
    text = _MULTISPACE_PATTERN.sub(" ", text)
    return text.strip()

def _merge_pair_in_word(symbols, a, b):
    out, i = [], 0
    while i < len(symbols):
        if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
            out.append(a + b); i += 2
        else:
            out.append(symbols[i]); i += 1
    return out

class ArabicFinancialTokenizer:
    def __init__(self, vocab_size=None):
        self.vocab_size = vocab_size or SENTIMENT_VOCAB_SIZE
        self.special_tokens = dict(SENTIMENT_SPECIAL_TOKENS)
        self.vocab = dict(self.special_tokens)
        self.merges = []
        self._merge_ranks = {}

    def train_bpe(self, corpus):
        word_freq = Counter()
        for text in corpus:
            for word in preprocess(text).split():
                word_freq[word] += 1
        if not word_freq:
            raise ValueError("train_bpe: corpus produced zero words after preprocessing")
        splits = {word: list(word) for word in word_freq}
        charset = sorted({ch for word in word_freq for ch in word})
        vocab = dict(self.special_tokens)
        next_id = len(vocab)
        for ch in charset:
            if ch not in vocab:
                vocab[ch] = next_id; next_id += 1
        merges = []
        while len(vocab) < self.vocab_size:
            pair_freq = Counter()
            for word, freq in word_freq.items():
                s = splits[word]
                for i in range(len(s) - 1):
                    pair_freq[(s[i], s[i + 1])] += freq
            if not pair_freq:
                break
            best_pair = max(pair_freq.items(), key=lambda kv: kv[1])[0]
            a, b = best_pair
            for word in splits:
                splits[word] = _merge_pair_in_word(splits[word], a, b)
            merges.append(best_pair)
            vocab[a + b] = next_id; next_id += 1
        self.vocab = vocab
        self.merges = merges
        self._merge_ranks = {pair: i for i, pair in enumerate(merges)}
        print(f"train_bpe: {len(merges)} merges, final vocab size {len(vocab)}")

    def _apply_merges(self, symbols):
        symbols = list(symbols)
        while len(symbols) > 1:
            pairs = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
            ranked = [(self._merge_ranks[p], i) for i, p in enumerate(pairs) if p in self._merge_ranks]
            if not ranked:
                break
            _, bi = min(ranked)
            a, b = symbols[bi], symbols[bi + 1]
            symbols = symbols[:bi] + [a + b] + symbols[bi + 2:]
        return symbols

    def encode(self, text, max_length=None):
        max_length = max_length or SENTIMENT_MAX_SEQ_LEN
        cls_id, sep_id = self.special_tokens["[CLS]"], self.special_tokens["[SEP]"]
        unk_id, pad_id = self.special_tokens["[UNK]"], self.special_tokens["[PAD]"]
        text = preprocess(text)
        ids = [cls_id]
        for word in text.split():
            for sym in self._apply_merges(list(word)):
                ids.append(self.vocab.get(sym, unk_id))
        ids.append(sep_id)
        if len(ids) > max_length:
            ids = ids[:max_length - 1] + [sep_id]
        mask = [1] * len(ids)
        pad = max_length - len(ids)
        if pad > 0:
            ids += [pad_id] * pad; mask += [0] * pad
        return {"input_ids": ids, "attention_mask": mask}

    def save(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"vocab": self.vocab, "merges": self.merges, "vocab_size": self.vocab_size}, f, ensure_ascii=False)

In [ ]:
# ============================================================
# TRANSFORMER  (verbatim from models/sentiment/model.py) — from scratch
# ============================================================
import math
import torch.nn as nn

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, : x.size(1), :]

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads; self.d_k = d_model // n_heads
        self.w_q = nn.Linear(d_model, d_model); self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model); self.w_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, attention_mask):
        B, T, D = x.shape
        q = self.w_q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = self.w_k(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = self.w_v(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_k)
        mask = attention_mask[:, None, None, :].bool()
        scores = scores.masked_fill(~mask, float("-inf"))
        attn = torch.softmax(scores, dim=-1)
        attn = torch.nan_to_num(attn, nan=0.0)
        attn = self.dropout(attn)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, D)
        return self.w_o(out)

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.linear2(self.dropout(torch.nn.functional.gelu(self.linear1(x))))

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, attention_mask):
        h = x + self.dropout(self.attn(self.norm1(x), attention_mask))
        return h + self.dropout(self.ffn(self.norm2(h)))

class AttentionPooling(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.score = nn.Linear(d_model, 1)
    def forward(self, x, attention_mask):
        scores = self.score(x).squeeze(-1)
        scores = scores.masked_fill(attention_mask == 0, float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        pooled = torch.einsum("bt,btd->bd", weights, x)
        return pooled, weights

class ArabicSentimentTransformer(nn.Module):
    def __init__(self, vocab_size=None, d_model=None, n_heads=None, n_layers=None,
                 d_ff=None, max_seq_len=None, dropout=None, n_classes=None):
        super().__init__()
        vocab_size  = vocab_size  or SENTIMENT_VOCAB_SIZE
        d_model     = d_model     or SENTIMENT_D_MODEL
        n_heads     = n_heads     or SENTIMENT_N_HEADS
        n_layers    = n_layers    or SENTIMENT_N_LAYERS
        d_ff        = d_ff        or SENTIMENT_D_FF
        max_seq_len = max_seq_len or SENTIMENT_MAX_SEQ_LEN
        dropout     = SENTIMENT_DROPOUT if dropout is None else dropout
        n_classes   = n_classes   or SENTIMENT_N_CLASSES
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=SENTIMENT_SPECIAL_TOKENS["[PAD]"])
        self.pos_encoding = SinusoidalPositionalEncoding(d_model, max_seq_len)
        self.embed_dropout = nn.Dropout(dropout)
        self.layers = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)
        self.pool = AttentionPooling(d_model)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 128), nn.GELU(), nn.Dropout(dropout), nn.Linear(128, n_classes))
    def forward(self, input_ids, attention_mask):
        x = self.embedding(input_ids) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        x = self.embed_dropout(x)
        for layer in self.layers:
            x = layer(x, attention_mask)
        x = self.final_norm(x)
        pooled, w = self.pool(x, attention_mask)
        return self.classifier(pooled), w

In [ ]:
# ============================================================
# TRAINING UTILITIES  (verbatim from models/sentiment/trainer.py)
# ============================================================
import numpy as np, pandas as pd
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

def compute_class_weights(labels, n_classes=None):
    n_classes = n_classes or SENTIMENT_N_CLASSES
    counts = np.bincount(np.asarray(labels, dtype=int), minlength=n_classes)
    counts_safe = np.where(counts == 0, 1, counts)
    return len(labels) / (n_classes * counts_safe)

def time_based_split(df, val_fraction=None):
    val_fraction = val_fraction if val_fraction is not None else SENTIMENT_VAL_FRACTION
    dated = df[df["date"].notna()].sort_values("date")
    undated = df[df["date"].isna()]
    if len(dated) < 10:
        print(f"time_based_split: only {len(dated)} dated rows -> no held-out validation")
        return pd.concat([dated, undated], ignore_index=True), dated.iloc[0:0]
    k = int(len(dated) * (1 - val_fraction))
    train_df = pd.concat([dated.iloc[:k], undated], ignore_index=True)
    return train_df, dated.iloc[k:].reset_index(drop=True)

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, weights, tokenizer, max_length=None):
        self.texts = list(texts); self.labels = list(labels); self.weights = list(weights)
        self.tokenizer = tokenizer; self.max_length = max_length or SENTIMENT_MAX_SEQ_LEN
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer.encode(self.texts[idx], max_length=self.max_length)
        return {"input_ids": torch.tensor(enc["input_ids"], dtype=torch.long),
                "attention_mask": torch.tensor(enc["attention_mask"], dtype=torch.long),
                "label": torch.tensor(self.labels[idx], dtype=torch.long),
                "weight": torch.tensor(self.weights[idx], dtype=torch.float)}

def warmup_cosine_schedule(optimizer, total_steps, warmup_ratio=None):
    warmup_ratio = warmup_ratio if warmup_ratio is not None else SENTIMENT_WARMUP_RATIO
    warmup_steps = max(1, int(total_steps * warmup_ratio))
    def lr_lambda(step):
        if step < warmup_steps:
            return step / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def confusion_matrix(y_true, y_pred, n_classes=None):
    n_classes = n_classes or SENTIMENT_N_CLASSES
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for t, p in zip(y_true, y_pred): cm[int(t), int(p)] += 1
    return cm

def precision_recall_f1_per_class(cm):
    n = cm.shape[0]; precision, recall, f1 = np.zeros(n), np.zeros(n), np.zeros(n)
    for c in range(n):
        tp = cm[c, c]; fp = cm[:, c].sum() - tp; fn = cm[c, :].sum() - tp
        precision[c] = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall[c]    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        d = precision[c] + recall[c]
        f1[c] = 2 * precision[c] * recall[c] / d if d > 0 else 0.0
    return precision, recall, f1

def macro_f1_score(y_true, y_pred, n_classes=None):
    _, _, f1 = precision_recall_f1_per_class(confusion_matrix(y_true, y_pred, n_classes))
    return float(f1.mean())

def expected_calibration_error(confidences, correctness, n_bins=10):
    confidences = np.asarray(confidences, float); correctness = np.asarray(correctness, float)
    n = len(confidences)
    if n == 0: return float("nan")
    edges = np.linspace(0, 1, n_bins + 1); ece = 0.0
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        mask = (confidences > lo) & (confidences <= hi) if i > 0 else (confidences >= lo) & (confidences <= hi)
        if mask.sum() == 0: continue
        ece += (mask.sum() / n) * abs(correctness[mask].mean() - confidences[mask].mean())
    return float(ece)

def _run_epoch(model, loader, class_weights_t, device, optimizer=None, scheduler=None, grad_clip=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, total_w = 0.0, 0.0
    all_true, all_pred, all_conf = [], [], []
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for batch in loader:
            input_ids = batch["input_ids"].to(device); attn = batch["attention_mask"].to(device)
            labels = batch["label"].to(device); sw = batch["weight"].to(device)
            logits, _ = model(input_ids, attn)
            per = F.cross_entropy(logits, labels, reduction="none")
            cw = class_weights_t[labels] * sw
            loss = (per * cw).sum() / cw.sum().clamp_min(1e-8)
            if is_train:
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip or SENTIMENT_GRAD_CLIP_NORM)
                optimizer.step()
                if scheduler is not None: scheduler.step()
            probs = torch.softmax(logits, dim=-1); conf, pred = probs.max(dim=-1)
            total_loss += loss.item() * cw.sum().item(); total_w += cw.sum().item()
            all_true += labels.cpu().tolist(); all_pred += pred.cpu().tolist()
            all_conf += conf.detach().cpu().tolist()
    ml = total_loss / total_w if total_w > 0 else float("nan")
    return ml, np.array(all_true), np.array(all_pred), np.array(all_conf)

In [ ]:
# ============================================================
# LOAD DATA + TRAIN BPE TOKENIZER
# ============================================================
df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
# normalize expected columns
if "date" not in df.columns: df["date"] = pd.NaT
if "weight" not in df.columns: df["weight"] = 1.0
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["label"] = df["label"].astype(int)
df["text"] = df["text"].astype(str)
print("Rows:", len(df))
print("Class balance:\n", df["label"].value_counts().sort_index())

tokenizer = ArabicFinancialTokenizer(vocab_size=SENTIMENT_VOCAB_SIZE)
tokenizer.train_bpe(df["text"].tolist())

train_df, val_df = time_based_split(df)
# If time-based split leaves no validation (few/undated rows), carve a small
# stratified holdout just for a sanity metric (synthetic bootstrap only).
if len(val_df) == 0 and len(train_df) >= 15:
    rng = np.random.default_rng(42)
    hold = []
    for c, g in train_df.groupby("label"):
        idx = g.index.to_numpy().copy(); rng.shuffle(idx)
        hold += list(idx[:max(1, int(0.15 * len(idx)))])
    val_df = train_df.loc[hold].reset_index(drop=True)
    train_df = train_df.drop(index=hold).reset_index(drop=True)
    print("(No dated rows -> using stratified random holdout for a sanity metric only.)")
print("Train:", len(train_df), "| Val:", len(val_df))

In [ ]:
# ============================================================
# TRAIN  (AdamW + warmup-cosine, weighted CE, grad clip)
# ============================================================
class_weights = compute_class_weights(train_df["label"].to_numpy())
class_weights_t = torch.tensor(class_weights, dtype=torch.float, device=DEVICE)
print("class weights:", class_weights)

train_ds = SentimentDataset(train_df["text"], train_df["label"], train_df["weight"], tokenizer)
train_loader = DataLoader(train_ds, batch_size=SENTIMENT_BATCH_SIZE, shuffle=True)
val_loader = None
if len(val_df) > 0:
    val_ds = SentimentDataset(val_df["text"], val_df["label"], val_df["weight"], tokenizer)
    val_loader = DataLoader(val_ds, batch_size=SENTIMENT_BATCH_SIZE, shuffle=False)

model = ArabicSentimentTransformer(vocab_size=len(tokenizer.vocab)).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model params: {n_params:,}")
optimizer = torch.optim.AdamW(model.parameters(), lr=SENTIMENT_LR, weight_decay=SENTIMENT_WEIGHT_DECAY)
total_steps = max(1, len(train_loader) * SENTIMENT_EPOCHS)
scheduler = warmup_cosine_schedule(optimizer, total_steps)

best_val_f1, best_state = -1.0, None
history = []
for epoch in range(SENTIMENT_EPOCHS):
    tr_loss, tr_true, tr_pred, _ = _run_epoch(model, train_loader, class_weights_t, DEVICE,
                                              optimizer=optimizer, scheduler=scheduler)
    tr_f1 = macro_f1_score(tr_true, tr_pred)
    va_loss, va_f1 = float("nan"), float("nan")
    if val_loader is not None:
        va_loss, va_true, va_pred, _ = _run_epoch(model, val_loader, class_weights_t, DEVICE)
        va_f1 = macro_f1_score(va_true, va_pred)
    history.append(dict(epoch=epoch, train_loss=tr_loss, train_f1=tr_f1, val_loss=va_loss, val_f1=va_f1))
    print(f"epoch {epoch+1:2d}/{SENTIMENT_EPOCHS}  train_loss={tr_loss:.4f} f1={tr_f1:.4f}  val_loss={va_loss:.4f} f1={va_f1:.4f}")
    # keep best (fall back to train f1 if no validation set)
    score = va_f1 if val_loader is not None else tr_f1
    if score >= best_val_f1:
        best_val_f1 = score
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

if best_state is not None:
    model.load_state_dict(best_state)
print("Best selection F1:", best_val_f1)

In [ ]:
# ============================================================
# EVALUATE  (macro F1, per-class P/R/F1, ECE)
# ============================================================
eval_df = val_df if len(val_df) > 0 else train_df
ev_ds = SentimentDataset(eval_df["text"], eval_df["label"], eval_df["weight"], tokenizer)
ev_loader = DataLoader(ev_ds, batch_size=SENTIMENT_BATCH_SIZE, shuffle=False)
ones = torch.ones(SENTIMENT_N_CLASSES, device=DEVICE)
_, y_true, y_pred, conf = _run_epoch(model, ev_loader, ones, DEVICE)
cm = confusion_matrix(y_true, y_pred)
p, r, f1 = precision_recall_f1_per_class(cm)
ece = expected_calibration_error(conf, (y_true == y_pred).astype(float))
names = ["NEG", "NEU", "POS"]
print("Confusion matrix (rows=true, cols=pred):\n", cm)
for i, nm in enumerate(names):
    print(f"  {nm}: precision={p[i]:.3f} recall={r[i]:.3f} f1={f1[i]:.3f}")
print(f"Macro F1: {f1.mean():.4f} | ECE: {ece:.4f}")

In [ ]:
# ============================================================
# SAVE CHECKPOINT  (exact keys SentimentPipeline._try_load_checkpoint reads)
# ============================================================
checkpoint = {
    "vocab_size": tokenizer.vocab_size,
    "tokenizer_vocab": tokenizer.vocab,
    "tokenizer_merges": tokenizer.merges,
    "model_state_dict": model.state_dict(),
    # extra metadata (ignored by loader, handy for provenance)
    "arch": {"d_model": SENTIMENT_D_MODEL, "n_heads": SENTIMENT_N_HEADS,
             "n_layers": SENTIMENT_N_LAYERS, "d_ff": SENTIMENT_D_FF,
             "max_seq_len": SENTIMENT_MAX_SEQ_LEN, "n_classes": SENTIMENT_N_CLASSES},
    "history": history,
    "best_f1": best_val_f1,
}
torch.save(checkpoint, OUTPUT_CHECKPOINT)
tokenizer.save(OUTPUT_TOKENIZER)
print("Saved:", OUTPUT_CHECKPOINT)
print("Saved:", OUTPUT_TOKENIZER)

# Sanity: reload exactly the way the repo pipeline does, confirm it works.
ckpt = torch.load(OUTPUT_CHECKPOINT, map_location="cpu")
tok2 = ArabicFinancialTokenizer(vocab_size=ckpt["vocab_size"])
tok2.vocab = ckpt["tokenizer_vocab"]
tok2.merges = [tuple(x) for x in ckpt["tokenizer_merges"]]
tok2._merge_ranks = {p: i for i, p in enumerate(tok2.merges)}
m2 = ArabicSentimentTransformer(vocab_size=len(tok2.vocab))
m2.load_state_dict(ckpt["model_state_dict"])   # strict=True — must match repo arch
m2.eval()
enc = tok2.encode("ارتفعت ارباح الشركة بشكل قوي")
with torch.no_grad():
    logits, _ = m2(torch.tensor([enc["input_ids"]]), torch.tensor([enc["attention_mask"]]))
    probs = torch.softmax(logits, -1)[0]
print("Reload OK. Sample probs [NEG, NEU, POS]:", [round(float(x), 3) for x in probs])
print("\n>>> Download best_sentiment_model.pt from the Output panel and place it at:")
print(">>> models/sentiment/checkpoints/best_sentiment_model.pt")